# Retraining Window Experiments and Performance Tracking

## Learning Objectives

In this notebook, you will learn:
- Different training data window strategies for model retraining
- How to implement post-drift focus, rolling window, and hybrid approaches
- How to optimize window size for different drift patterns
- How to track model performance after retraining
- How to make data-driven retraining decisions

## Introduction

When drift is detected, one of the most critical decisions is selecting the appropriate training data window for retraining. The choice of windowing strategy can significantly impact model performance.

### Training Data Window Strategies

1. **Post-Drift Focus**: Use only data after the drift point
   - Best for: Abrupt, permanent drift
   - Pros: Adapts quickly to new distribution
   - Cons: May lose knowledge of rare events

2. **Rolling/Sliding Window**: Use a fixed-size window of recent data
   - Best for: Gradual or seasonal drift
   - Pros: Balances old and new knowledge
   - Cons: Requires window size tuning

3. **Hybrid Approach**: Combine historical core with recent window
   - Best for: Preserving rare events while adapting
   - Pros: Best of both worlds
   - Cons: More complex to implement

4. **Event-Bounded Windows**: Align with business events
   - Best for: Known business cycle changes
   - Pros: Aligned with domain knowledge
   - Cons: Requires domain expertise

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Generate Synthetic Classification Data with Drift

In [ ]:
def generate_classification_data_with_drift(n_samples_per_period=500, n_periods=20, 
                                           drift_period=10, drift_type='abrupt'):
    """
    Generate synthetic classification data with concept drift.
    
    Parameters:
    -----------
    n_samples_per_period : int
        Number of samples per time period
    n_periods : int
        Total number of time periods
    drift_period : int
        Period where drift occurs
    drift_type : str
        Type of drift: 'abrupt', 'gradual', or 'recurring'
    
    Returns:
    --------
    data : DataFrame
        Complete dataset with features, labels, and time periods
    """
    all_data = []
    
    for period in range(n_periods):
        # Generate features
        X1 = np.random.normal(0, 1, n_samples_per_period)
        X2 = np.random.normal(0, 1, n_samples_per_period)
        
        # Generate labels based on drift type
        if drift_type == 'abrupt':
            if period < drift_period:
                # Pre-drift: Y depends on X1 + X2
                decision_boundary = X1 + X2
            else:
                # Post-drift: Y depends on X1 - X2 (concept change)
                decision_boundary = X1 - X2
        
        elif drift_type == 'gradual':
            if period < drift_period:
                decision_boundary = X1 + X2
            elif period < drift_period + 5:
                # Gradual transition
                progress = (period - drift_period) / 5
                decision_boundary = (1 - progress) * (X1 + X2) + progress * (X1 - X2)
            else:
                decision_boundary = X1 - X2
        
        elif drift_type == 'recurring':
            # Alternating concept every 5 periods
            if (period // 5) % 2 == 0:
                decision_boundary = X1 + X2
            else:
                decision_boundary = X1 - X2
        
        # Add noise and create binary labels
        noise = np.random.normal(0, 0.5, n_samples_per_period)
        y = (decision_boundary + noise > 0).astype(int)
        
        # Create period data
        period_data = pd.DataFrame({
            'X1': X1,
            'X2': X2,
            'y': y,
            'period': period
        })
        
        all_data.append(period_data)
    
    return pd.concat(all_data, ignore_index=True)

# Generate data with abrupt drift
data = generate_classification_data_with_drift(n_samples_per_period=500, n_periods=20, 
                                              drift_period=10, drift_type='abrupt')

print(f"Generated {len(data)} samples across {data['period'].nunique()} periods")
print(f"Drift occurs at period {10}")
print(f"\nData preview:")
print(data.head())
print(f"\nClass distribution: {data['y'].value_counts().to_dict()}")

## 2. Visualize Data and Drift

In [ ]:
# Visualize decision boundary change
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pre-drift (period 5)
pre_drift = data[data['period'] == 5]
axes[0].scatter(pre_drift[pre_drift['y']==0]['X1'], pre_drift[pre_drift['y']==0]['X2'], 
               alpha=0.5, label='Class 0', s=20)
axes[0].scatter(pre_drift[pre_drift['y']==1]['X1'], pre_drift[pre_drift['y']==1]['X2'], 
               alpha=0.5, label='Class 1', s=20)
axes[0].set_xlabel('X1')
axes[0].set_ylabel('X2')
axes[0].set_title('Pre-Drift (Period 5)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drift period (period 10)
drift_period_data = data[data['period'] == 10]
axes[1].scatter(drift_period_data[drift_period_data['y']==0]['X1'], 
               drift_period_data[drift_period_data['y']==0]['X2'], 
               alpha=0.5, label='Class 0', s=20)
axes[1].scatter(drift_period_data[drift_period_data['y']==1]['X1'], 
               drift_period_data[drift_period_data['y']==1]['X2'], 
               alpha=0.5, label='Class 1', s=20)
axes[1].set_xlabel('X1')
axes[1].set_ylabel('X2')
axes[1].set_title('Drift Period (Period 10)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Post-drift (period 15)
post_drift = data[data['period'] == 15]
axes[2].scatter(post_drift[post_drift['y']==0]['X1'], post_drift[post_drift['y']==0]['X2'], 
               alpha=0.5, label='Class 0', s=20)
axes[2].scatter(post_drift[post_drift['y']==1]['X1'], post_drift[post_drift['y']==1]['X2'], 
               alpha=0.5, label='Class 1', s=20)
axes[2].set_xlabel('X1')
axes[2].set_ylabel('X2')
axes[2].set_title('Post-Drift (Period 15)', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Implement Different Windowing Strategies

In [ ]:
def train_model_with_window(data, window_strategy, window_size=None, drift_period=10):
    """
    Train model using different windowing strategies and track performance.
    
    Parameters:
    -----------
    data : DataFrame
        Complete dataset
    window_strategy : str
        'all_data', 'post_drift', 'rolling', or 'hybrid'
    window_size : int
        Size of rolling window (in periods)
    drift_period : int
        Period where drift occurs
    
    Returns:
    --------
    performance : DataFrame
        Performance metrics over time
    """
    n_periods = data['period'].max() + 1
    performance_records = []
    
    for test_period in range(drift_period, n_periods):
        # Select training data based on strategy
        if window_strategy == 'all_data':
            # Use all historical data
            train_data = data[data['period'] < test_period]
        
        elif window_strategy == 'post_drift':
            # Use only post-drift data
            train_data = data[(data['period'] >= drift_period) & (data['period'] < test_period)]
        
        elif window_strategy == 'rolling':
            # Use rolling window
            start_period = max(0, test_period - window_size)
            train_data = data[(data['period'] >= start_period) & (data['period'] < test_period)]
        
        elif window_strategy == 'hybrid':
            # Use pre-drift core + recent window
            core_data = data[data['period'] < drift_period]
            start_period = max(drift_period, test_period - window_size)
            recent_data = data[(data['period'] >= start_period) & (data['period'] < test_period)]
            train_data = pd.concat([core_data, recent_data])
        
        # Skip if insufficient training data
        if len(train_data) < 100:
            continue
        
        # Prepare training and test sets
        X_train = train_data[['X1', 'X2']].values
        y_train = train_data['y'].values
        
        test_data = data[data['period'] == test_period]
        X_test = test_data[['X1', 'X2']].values
        y_test = test_data['y'].values
        
        # Train model
        model = LogisticRegression(random_state=42, max_iter=1000)
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)
        
        performance_records.append({
            'test_period': test_period,
            'train_size': len(train_data),
            'accuracy': accuracy,
            'f1_score': f1,
            'auc': auc
        })
    
    return pd.DataFrame(performance_records)

# Test different strategies
print("Training models with different windowing strategies...")
print("=" * 60)

perf_all_data = train_model_with_window(data, 'all_data')
print("✓ All data strategy completed")

perf_post_drift = train_model_with_window(data, 'post_drift')
print("✓ Post-drift strategy completed")

perf_rolling_3 = train_model_with_window(data, 'rolling', window_size=3)
print("✓ Rolling window (3 periods) completed")

perf_rolling_5 = train_model_with_window(data, 'rolling', window_size=5)
print("✓ Rolling window (5 periods) completed")

perf_hybrid = train_model_with_window(data, 'hybrid', window_size=3)
print("✓ Hybrid strategy completed")

print("\nAll strategies completed successfully!")

## 4. Compare Performance Across Strategies

In [ ]:
# Plot performance comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy comparison
axes[0, 0].plot(perf_all_data['test_period'], perf_all_data['accuracy'], 
               marker='o', label='All Data', linewidth=2)
axes[0, 0].plot(perf_post_drift['test_period'], perf_post_drift['accuracy'], 
               marker='s', label='Post-Drift', linewidth=2)
axes[0, 0].plot(perf_rolling_3['test_period'], perf_rolling_3['accuracy'], 
               marker='^', label='Rolling (3 periods)', linewidth=2)
axes[0, 0].plot(perf_rolling_5['test_period'], perf_rolling_5['accuracy'], 
               marker='v', label='Rolling (5 periods)', linewidth=2)
axes[0, 0].plot(perf_hybrid['test_period'], perf_hybrid['accuracy'], 
               marker='d', label='Hybrid', linewidth=2)
axes[0, 0].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='Drift Point')
axes[0, 0].set_xlabel('Test Period')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# F1 Score comparison
axes[0, 1].plot(perf_all_data['test_period'], perf_all_data['f1_score'], 
               marker='o', label='All Data', linewidth=2)
axes[0, 1].plot(perf_post_drift['test_period'], perf_post_drift['f1_score'], 
               marker='s', label='Post-Drift', linewidth=2)
axes[0, 1].plot(perf_rolling_3['test_period'], perf_rolling_3['f1_score'], 
               marker='^', label='Rolling (3 periods)', linewidth=2)
axes[0, 1].plot(perf_rolling_5['test_period'], perf_rolling_5['f1_score'], 
               marker='v', label='Rolling (5 periods)', linewidth=2)
axes[0, 1].plot(perf_hybrid['test_period'], perf_hybrid['f1_score'], 
               marker='d', label='Hybrid', linewidth=2)
axes[0, 1].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='Drift Point')
axes[0, 1].set_xlabel('Test Period')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('F1 Score Comparison', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AUC comparison
axes[1, 0].plot(perf_all_data['test_period'], perf_all_data['auc'], 
               marker='o', label='All Data', linewidth=2)
axes[1, 0].plot(perf_post_drift['test_period'], perf_post_drift['auc'], 
               marker='s', label='Post-Drift', linewidth=2)
axes[1, 0].plot(perf_rolling_3['test_period'], perf_rolling_3['auc'], 
               marker='^', label='Rolling (3 periods)', linewidth=2)
axes[1, 0].plot(perf_rolling_5['test_period'], perf_rolling_5['auc'], 
               marker='v', label='Rolling (5 periods)', linewidth=2)
axes[1, 0].plot(perf_hybrid['test_period'], perf_hybrid['auc'], 
               marker='d', label='Hybrid', linewidth=2)
axes[1, 0].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='Drift Point')
axes[1, 0].set_xlabel('Test Period')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].set_title('AUC Comparison', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Training set size comparison
axes[1, 1].plot(perf_all_data['test_period'], perf_all_data['train_size'], 
               marker='o', label='All Data', linewidth=2)
axes[1, 1].plot(perf_post_drift['test_period'], perf_post_drift['train_size'], 
               marker='s', label='Post-Drift', linewidth=2)
axes[1, 1].plot(perf_rolling_3['test_period'], perf_rolling_3['train_size'], 
               marker='^', label='Rolling (3 periods)', linewidth=2)
axes[1, 1].plot(perf_rolling_5['test_period'], perf_rolling_5['train_size'], 
               marker='v', label='Rolling (5 periods)', linewidth=2)
axes[1, 1].plot(perf_hybrid['test_period'], perf_hybrid['train_size'], 
               marker='d', label='Hybrid', linewidth=2)
axes[1, 1].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='Drift Point')
axes[1, 1].set_xlabel('Test Period')
axes[1, 1].set_ylabel('Training Set Size')
axes[1, 1].set_title('Training Set Size', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary Statistics

In [ ]:
# Calculate average performance metrics
summary = pd.DataFrame({
    'Strategy': ['All Data', 'Post-Drift', 'Rolling (3)', 'Rolling (5)', 'Hybrid'],
    'Avg Accuracy': [
        perf_all_data['accuracy'].mean(),
        perf_post_drift['accuracy'].mean(),
        perf_rolling_3['accuracy'].mean(),
        perf_rolling_5['accuracy'].mean(),
        perf_hybrid['accuracy'].mean()
    ],
    'Avg F1': [
        perf_all_data['f1_score'].mean(),
        perf_post_drift['f1_score'].mean(),
        perf_rolling_3['f1_score'].mean(),
        perf_rolling_5['f1_score'].mean(),
        perf_hybrid['f1_score'].mean()
    ],
    'Avg AUC': [
        perf_all_data['auc'].mean(),
        perf_post_drift['auc'].mean(),
        perf_rolling_3['auc'].mean(),
        perf_rolling_5['auc'].mean(),
        perf_hybrid['auc'].mean()
    ],
    'Avg Train Size': [
        perf_all_data['train_size'].mean(),
        perf_post_drift['train_size'].mean(),
        perf_rolling_3['train_size'].mean(),
        perf_rolling_5['train_size'].mean(),
        perf_hybrid['train_size'].mean()
    ]
})

print("\nPerformance Summary (Post-Drift Periods):")
print("=" * 80)
print(summary.to_string(index=False))

# Find best strategy
best_accuracy_idx = summary['Avg Accuracy'].idxmax()
best_f1_idx = summary['Avg F1'].idxmax()

print("\n" + "=" * 80)
print(f"Best Strategy (Accuracy): {summary.loc[best_accuracy_idx, 'Strategy']} "
      f"({summary.loc[best_accuracy_idx, 'Avg Accuracy']:.4f})")
print(f"Best Strategy (F1 Score): {summary.loc[best_f1_idx, 'Strategy']} "
      f"({summary.loc[best_f1_idx, 'Avg F1']:.4f})")
print("=" * 80)

## 6. Window Size Optimization

In [ ]:
# Test different window sizes
window_sizes = [2, 3, 4, 5, 6, 7, 8]
window_performance = []

print("Testing different window sizes...")
for window_size in window_sizes:
    perf = train_model_with_window(data, 'rolling', window_size=window_size)
    window_performance.append({
        'window_size': window_size,
        'avg_accuracy': perf['accuracy'].mean(),
        'avg_f1': perf['f1_score'].mean(),
        'avg_auc': perf['auc'].mean()
    })
    print(f"  Window size {window_size}: Accuracy = {perf['accuracy'].mean():.4f}")

window_perf_df = pd.DataFrame(window_performance)

# Plot window size optimization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy vs window size
axes[0].plot(window_perf_df['window_size'], window_perf_df['avg_accuracy'], 
            marker='o', linewidth=2, markersize=8)
optimal_window = window_perf_df.loc[window_perf_df['avg_accuracy'].idxmax(), 'window_size']
axes[0].axvline(x=optimal_window, color='red', linestyle='--', 
               label=f'Optimal: {optimal_window}')
axes[0].set_xlabel('Window Size (periods)')
axes[0].set_ylabel('Average Accuracy')
axes[0].set_title('Accuracy vs Window Size', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 vs window size
axes[1].plot(window_perf_df['window_size'], window_perf_df['avg_f1'], 
            marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_xlabel('Window Size (periods)')
axes[1].set_ylabel('Average F1 Score')
axes[1].set_title('F1 Score vs Window Size', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# AUC vs window size
axes[2].plot(window_perf_df['window_size'], window_perf_df['avg_auc'], 
            marker='^', linewidth=2, markersize=8, color='purple')
axes[2].set_xlabel('Window Size (periods)')
axes[2].set_ylabel('Average AUC')
axes[2].set_title('AUC vs Window Size', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimal window size: {optimal_window} periods")

## Key Takeaways

### Strategy Selection Guidelines

| Drift Type | Recommended Strategy | Rationale |
|------------|---------------------|----------|
| Abrupt, permanent | Post-drift focus | Quickly adapts to new distribution |
| Gradual | Rolling window (3-5 periods) | Balances adaptation and stability |
| Seasonal/recurring | Rolling window (aligned with cycle) | Captures recurring patterns |
| With rare events | Hybrid approach | Preserves rare event knowledge |
| Unknown | Rolling window (moderate size) | Safe default choice |

### Window Size Optimization

1. **Too small** (1-2 periods):
   - Pros: Fast adaptation
   - Cons: High variance, may overfit to noise

2. **Moderate** (3-6 periods):
   - Pros: Good balance, stable performance
   - Cons: May lag behind rapid drift

3. **Too large** (10+ periods):
   - Pros: Low variance, captures long-term patterns
   - Cons: Slow adaptation, includes stale data

### Best Practices

1. **Start with rolling window**: Use 3-5 periods as default
2. **Validate on historical data**: Test different strategies on past drift events
3. **Monitor performance**: Track metrics after each retraining
4. **Consider business cycles**: Align window with domain knowledge
5. **Preserve rare events**: Use hybrid approach when rare events are important
6. **Automate retraining**: Set up pipelines triggered by drift detection
7. **A/B test strategies**: Compare strategies in production

### Practical Recommendations

**For Production Systems:**
- Use rolling window as default (3-6 months for most applications)
- Implement automated window size tuning based on validation performance
- Maintain a champion-challenger framework to test new strategies
- Document retraining decisions and outcomes for continuous improvement

**For High-Stakes Applications:**
- Use hybrid approach to preserve rare but important events
- Implement multi-model ensemble with different window strategies
- Require human review before deploying retrained models
- Maintain rollback capability to previous model versions

## Exercises

1. Implement an adaptive window size strategy that automatically adjusts based on drift severity.

2. Create a weighted window approach where recent data has higher weight than older data.

3. Design an experiment to compare windowing strategies on gradual vs abrupt drift.

4. Build a retraining decision system that considers both drift detection and model performance.